# Part 5: ARIMA Baseline
**Project:** ML-Based Predictive Load Balancing and Auto-Scaling in Cloud Environments

A classical forecasting baseline for the LSTM. This notebook is self-contained: it rebuilds the same 5-minute request series and the same 70/15/15 split as the LSTM notebook, so all models are compared on exactly the same 1,192 test intervals.

1. The ARIMA order is chosen by AIC on the **training data only** (p in {1,2,3}, d in {0,1}, q in {0,1,2}).
2. The parameters are then kept fixed, and one-step-ahead (5-minute) forecasts are made for the test period from the observed values, the same information the LSTM gets.
3. The forecasts are saved as `predictions_arima.csv` in the format the CloudSim Plus simulation reads.

Runs on a normal Colab CPU runtime in about a minute.

## 1. Data, split and ARIMA forecasts

In [ ]:
import os, re, gzip, warnings, itertools
import numpy as np, pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.arima.model import ARIMA
warnings.filterwarnings('ignore')

# 1. Get the NASA log
FNAME = 'NASA_access_log_Jul95.gz'
if not os.path.exists(FNAME):
    !wget -q --timeout=60 ftp://ita.ee.lbl.gov/traces/NASA_access_log_Jul95.gz
if not os.path.exists(FNAME) or os.path.getsize(FNAME) < 1_000_000:
    print('Download failed, please upload NASA_access_log_Jul95.gz')
    from google.colab import files
    up = files.upload(); FNAME = list(up.keys())[0]

# 2. Requests per 5 minutes (same as the LSTM notebook)
ts_re = re.compile(r'\[(\d{2}/\w{3}/\d{4}:\d{2}:\d{2}:\d{2})')
stamps = []
opener = gzip.open if FNAME.endswith('.gz') else open
with opener(FNAME, 'rt', encoding='latin-1', errors='ignore') as f:
    for line in f:
        m = ts_re.search(line)
        if m: stamps.append(m.group(1))
ts = pd.to_datetime(pd.Series(stamps), format='%d/%b/%Y:%H:%M:%S', errors='coerce').dropna()
series = ts.dt.floor('5min').value_counts().sort_index()
series = series.reindex(pd.date_range(series.index.min(), series.index.max(), freq='5min'), fill_value=0).astype(float)

y = series.to_numpy()
n = len(y); n_train, n_val = int(n * 0.70), int(n * 0.15)
print('Intervals:', n, '| train', n_train, '| val', n_val, '| test', n - n_train - n_val)

def metrics(a, p):
    mask = a > 0
    return (round(mean_absolute_error(a, p), 2), round(np.sqrt(mean_squared_error(a, p)), 2),
            round(np.mean(np.abs((a[mask] - p[mask]) / a[mask])) * 100, 2))

# 3. ARIMA: order chosen by AIC on training data, then one-step-ahead forecasts on the test set
best = None
for p, d, q in itertools.product([1, 2, 3], [0, 1], [0, 1, 2]):
    try:
        res = ARIMA(y[:n_train], order=(p, d, q)).fit()
        if best is None or res.aic < best[1]:
            best = ((p, d, q), res.aic, res)
    except Exception:
        pass
order, aic, res = best
print('Best ARIMA order:', order, '| AIC:', round(aic, 1))

full = res.apply(y)                               # same parameters, whole series
arima_pred = np.clip(full.predict(start=n_train + n_val, end=n - 1), 0, None)
arima_metrics = pd.DataFrame([metrics(y[n_train + n_val:], arima_pred)],
                             index=[f'ARIMA{order}'], columns=['MAE', 'RMSE', 'MAPE (%)'])
print(arima_metrics)

## 2. Save forecasts for CloudSim Plus

Same columns as the LSTM notebook's `predictions.csv`. The ARIMA forecast goes in the `lstm_pred` column so the simulation can read it without code changes: in the CloudSim output for this file, the **PREDICTIVE_LSTM row is the ARIMA-driven policy**.

In [ ]:
os.makedirs('outputs', exist_ok=True)
arima_metrics.to_csv('outputs/arima_metrics.csv')

s = pd.Series(y)
test = slice(n_train + n_val, n)
out = pd.DataFrame({
    'step': np.arange(n - n_train - n_val),
    'timestamp': series.index[test].astype(str),
    'actual_requests': y[test].round().astype(int),
    'lstm_pred': arima_pred.round().astype(int),          # ARIMA forecast (column name kept for CloudSim)
    'ma_pred': s.rolling(12).mean().shift(1).to_numpy()[test].round().astype(int),
    'naive_pred': s.shift(1).to_numpy()[test].round().astype(int),
})
out.to_csv('outputs/predictions_arima.csv', index=False)
print(out.head())

from google.colab import files
files.download('outputs/predictions_arima.csv')